# Published detector on Conakry

Rußwurm et al. (2023) trained a deep model on MARIDA and other sets, with many hard negatives. Their weights are public, so this tests one question: does it stay off Guinea's tidal flats, where the other methods fail?

MariNeXt (Kikaki et al. 2024) is left out. Its code covers training and evaluation on its own dataset, with no ready way to run it on a new image.

Install the detector in its own environment, so that it does not change the versions the other scripts rely on. In a terminal:

```
python -m venv D:\guinea_data\mdd_env
D:\guinea_data\mdd_env\Scripts\pip install marinedebrisdetector
```

In [ ]:
from pathlib import Path
import json
import rasterio
import pandas as pd

ROOT = Path(r"D:\Guinea_Methodology\plasticwaste_experiments")
%cd {ROOT}
MDD = r"D:\guinea_data\mdd_env\Scripts\marinedebrisdetector"

## 1. Try it on its own example

This confirms the install works and shows the band layout it expects. Its example has about 300 MB.

In [ ]:
%cd 7_published_detector/outputs
!{MDD} --download-durban
!{MDD} durban_20190424.tif
%cd {ROOT}
with rasterio.open(ROOT / "7_published_detector/outputs/durban_20190424.tif") as s:
    print("bands", s.count, "type", s.dtypes[0], "max of band 4", s.read(4, window=((0, 500), (0, 500))).max())

**Check before going on.** It should show 12 bands. If it shows 13, or its values are not in the 0 to 10,000 range, tell me: the download in step 2 has to match it.

## 2. Download Conakry, two dates, about 600 MB each

In [ ]:
!python 7_published_detector/01_download_conakry.py

## 3. Run the detector

In [ ]:
%cd 7_published_detector/outputs
!{MDD} conakry_2024-01-11.tif
!{MDD} conakry_2024-01-26.tif
%cd {ROOT}

## 4. Score it against the tide mask

In [ ]:
!python 7_published_detector/02_score_predictions.py
display(pd.read_csv(ROOT / "7_published_detector/outputs/02_scores.csv").round(3))
json.loads((ROOT / "7_published_detector/outputs/02_rule.json").read_text())

**If `worth_adding` is true:** it is worth a closer look on more dates. It would still produce candidates, because no confirmed debris exists for these waters. If false, the current method stays, and the methodology can state that a published deep detector was tested too.